In [ ]:
from squiggs.neuron_viewer import NeuronViewer

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
from pathlib import Path

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"  # "MM012"  # "MR82"
sess_id = "20251028_140930"  # "20231219_130847"  # "20251027_152036"

In [ ]:
import pickle
import numpy as np
from utils.paths import MODELS_DIR

epoch = "choice"
tv = "response"
reg = "DLS"

betas = {strategy: [] for strategy in ["mb", "mf"]}

# load the families
file_path = (
    MODELS_DIR
    / "fit"
    / subj_id
    / sess_id
    / "tributaries"
    / "balance_and_norm"
    / "results_dict.pkl"
)

with open(file_path, "rb") as f:
    res_dict = pickle.load(f)


families_mb = res_dict[reg][epoch]["mb"]["families"]
families_mf = res_dict[reg][epoch]["mf"]["families"]
tv_labels = []

for seed in range(5):
    family_mb = families_mb[seed]
    family_mf = families_mf[seed]

    beta_mb = family_mb.mod_taskvar.tv.weight.data[:]
    beta_mf = family_mf.mod_taskvar.tv.weight.data[:]

    tv_idxs = []
    counter = 0
    for tv_ in family_mb.task_vars:
        for val in family_mb.trial_data[tv_].unique():
            if tv_ == tv:
                tv_idxs.append(counter)
                if tv_ == "response":
                    if val == 1:
                        val_str = "left"
                    elif val == -1:
                        val_str = "right"
                if tv_ == "rewarded":
                    if val == 1:
                        val_str = "correct"
                    elif val == 0:
                        val_str = "incorrect"
                if seed == 0:
                    tv_labels.append(f"{tv_}_{val_str}")
            counter += 1
    betas["mb"].append(np.array([beta_mb[tv_idx] for tv_idx in tv_idxs]))
    betas["mf"].append(np.array([beta_mf[tv_idx] for tv_idx in tv_idxs]))

betas["mb"] = np.array(betas["mb"])
betas["mf"] = np.array(betas["mf"])

In [ ]:
unit_idxs = []
for unit_idx in range(betas["mb"].shape[2]):
    for i, tv_idx in enumerate(tv_idxs):
        # first condition is mb ~ 0 and mf > 0
        if (
            np.isclose(np.mean(betas["mb"][:, i, unit_idx]), [0], rtol=1, atol=0.1)
            and np.mean(betas["mf"][:, i, unit_idx]) > 0.2
        ):
            print(
                tv_labels[i],
                unit_idx,
                np.mean(betas["mb"][:, i, unit_idx]),
                np.mean(betas["mf"][:, i, unit_idx]),
            )
            unit_idxs.append(unit_idx)

        # second condition is mf ~ 0 and mb > 0
        elif (
            np.isclose(np.mean(betas["mf"][:, i, unit_idx]), [0], rtol=1, atol=0.1)
            and np.mean(betas["mb"][:, i, unit_idx]) > 0.2
        ):
            print(
                tv_labels[i],
                unit_idx,
                np.mean(betas["mb"][:, i, unit_idx]),
                np.mean(betas["mf"][:, i, unit_idx]),
            )
            unit_idxs.append(unit_idx)

In [ ]:
from squiggs.renderers import PETHRasterRenderer
from core.data import get_psths_cond, get_choice_ts
from utils.paths import FIGURES_DIR

mode = tv

renderer = PETHRasterRenderer(
    event_times=get_choice_ts(family_mb.trial_data, mode=mode),
    spike_times=family_mb.spike_times[reg],
    peths=get_psths_cond(family_mb.psths[reg], family_mb.trial_data, mode=mode),
    pres=0.5,
    posts=0.5,
    binwidth_s=25 / 1000,
    s=0.2,
    linewidths=0.2,
    save_subdir=Path("peths") / subj_id / sess_id / reg / mode / "mb",
)

nv = NeuronViewer(
    num_units=family_mb.psths[reg].shape[0], render_func=renderer, fig_dir=FIGURES_DIR
)

In [ ]:
renderer = PETHRasterRenderer(
    event_times=get_choice_ts(family_mf.trial_data, mode=mode),
    spike_times=family_mf.spike_times[reg],
    peths=get_psths_cond(family_mf.psths[reg], family_mf.trial_data, mode=mode),
    pres=0.5,
    posts=0.5,
    binwidth_s=25 / 1000,
    s=0.2,
    linewidths=0.2,
    save_subdir=Path("peths") / subj_id / sess_id / reg / mode / "mf",
)

nv = NeuronViewer(
    num_units=family_mf.psths[reg].shape[0], render_func=renderer, fig_dir=FIGURES_DIR
)